# 05. Horizontal geological sections

This notebook plots horizontal intersections through the geological model at 980 and 1000 m elevation. Each section is calculated from the shared triangle mesh and the reconstructed interface elevations.

**Input:** `demo_model.npz`, generated by `run_model.py`. The input directory is selected by `HG3D_MODEL_DIR` (default: `outputs/model/`).

**Output:** `horizontal section.png`. Figures are displayed and saved to `HG3D_FIGURE_DIR` (default: `outputs/figures/`).

**Interpretation:** `ELEVATION_LEVELS_M` selects the two section elevations. Areas without unit presence or finite bounding elevations remain blank. XY coordinates use the same scale on both axes.


## 1. Imports and configuration


In [ ]:
import os
import tempfile
from pathlib import Path

# Matplotlib needs a writable cache directory before its first import.
MPLCONFIG_DIR = Path(tempfile.gettempdir()) / "matplotlib-harmonicgeo3d"
MPLCONFIG_DIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(MPLCONFIG_DIR))

# Project paths.
PROJECT_ROOT = next(
    (
        path
        for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
        if (path / "run_model.py").is_file() and (path / "notebooks").is_dir()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Start this notebook from the HarmonicGeo3D directory or a subdirectory."
    )

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.collections import PolyCollection
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

# Input data.
MODEL_DATA_DIR = Path(
    os.environ.get("HG3D_MODEL_DIR", PROJECT_ROOT / "outputs" / "model")
)
MODEL_PATH = MODEL_DATA_DIR / "demo_model.npz"

FONT_FAMILY = ["Arial", "DejaVu Sans"]
ELEVATION_LEVELS_M = (
    980.0,
    1000.0,
)  # Elevations of the left and right horizontal sections (m).
# Unit labels and colours follow body_names from bottom to top.
BODY_LABELS = ["Coal", "Lacustrine", "Deltaic", "Fluvial", "Topsoil"]
BODY_COLORS = ["#41464B", "#849EAF", "#88AAA3", "#C7AE80", "#B8B0A3"]

LEVEL_TOLERANCE_M = 1.0e-7
FIGURE_WIDTH_CM, FIGURE_HEIGHT_CM = 11, 7
CM_PER_INCH = 2.54
PLOT_RC = {
    "font.family": FONT_FAMILY,
    "axes.unicode_minus": False,
    "font.size": 9,
    "axes.labelsize": 6,
    "axes.titlesize": 7,
    "legend.fontsize": 4.2,
    "figure.dpi": 120,
    "savefig.dpi": 600,
}

# Figure output.
FIGURE_DIR = Path(
    os.environ.get("HG3D_FIGURE_DIR", PROJECT_ROOT / "outputs" / "figures")
)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

## 2. Data loading and validation


In [ ]:
with np.load(MODEL_PATH, allow_pickle=False) as model_data:
    initial_nodes_xy = np.asarray(model_data["initial_nodes_xy"], dtype=float)
    drill_names = np.asarray(model_data["drill_names"])
    final_nodes_xy = np.asarray(model_data["final_nodes_xy"], dtype=float)
    final_triangles = np.asarray(model_data["final_triangles"], dtype=int)
    interfaces_z = np.asarray(model_data["Z"], dtype=float)
    cell_presence_mask = np.asarray(model_data["cell_presence_mask"], dtype=bool)
    original_node_count = int(np.asarray(model_data["original_node_count"]))

n_interfaces = interfaces_z.shape[1]
n_bodies = n_interfaces - 1
if final_nodes_xy.shape != (interfaces_z.shape[0], 2):
    raise ValueError("final_nodes_xy and Z must have the same node count.")
if final_triangles.ndim != 2 or final_triangles.shape[1] != 3:
    raise ValueError("final_triangles must have shape (n_triangle, 3).")
if cell_presence_mask.shape != (len(final_triangles), n_bodies):
    raise ValueError(
        "cell_presence_mask must match the triangle and geological-unit counts."
    )
if len(BODY_LABELS) != n_bodies or len(BODY_COLORS) != n_bodies:
    raise ValueError(
        f"The legend configuration must match the {n_bodies} geological units in the model."
    )
if initial_nodes_xy.shape[0] != original_node_count:
    raise ValueError("The initial-node count must match original_node_count.")
if not np.allclose(initial_nodes_xy, final_nodes_xy[:original_node_count]):
    raise ValueError(
        "The first original_node_count rows of final_nodes_xy must match initial_nodes_xy."
    )
if not np.isfinite(final_nodes_xy).all():
    raise ValueError("final_nodes_xy must contain only finite coordinates.")

x_min_m, y_min_m = final_nodes_xy.min(axis=0)
x_max_m, y_max_m = final_nodes_xy.max(axis=0)
z_min_m = float(np.nanmin(interfaces_z))
z_max_m = float(np.nanmax(interfaces_z))
if len(ELEVATION_LEVELS_M) != 2:
    raise ValueError("ELEVATION_LEVELS_M must contain exactly two elevations.")
for level in ELEVATION_LEVELS_M:
    if not np.isfinite(level) or not (
        z_min_m - LEVEL_TOLERANCE_M <= level <= z_max_m + LEVEL_TOLERANCE_M
    ):
        raise ValueError(
            f"Section elevation {level} must be finite and within [{z_min_m:.3f}, {z_max_m:.3f}] m."
        )

## 3. Horizontal-intersection helpers


In [ ]:
def clip_polygon_by_linear_field(
    points: np.ndarray, values: np.ndarray, tolerance: float = LEVEL_TOLERANCE_M
) -> np.ndarray:
    """Clip a polygon to the nonnegative side of a linear field.

    Parameters
    ----------
    points : ndarray, shape (N, 2)
        Ordered polygon vertices in local XY coordinates, in metres.
    values : ndarray, shape (N,)
        Signed elevation differences at the vertices, in metres.
    tolerance : float, optional
        Elevation tolerance for classifying vertices near the clipping level.

    Returns
    -------
    ndarray, shape (M, 2)
        Vertices of the clipped polygon. An empty (0, 2) array is returned
        when fewer than three distinct vertices remain.
    """
    points = [np.asarray(point, dtype=float) for point in points]
    values = [float(value) for value in values]
    clipped = []
    for index, start in enumerate(points):
        end = points[(index + 1) % len(points)]
        value_start = values[index]
        value_end = values[(index + 1) % len(values)]
        start_inside = value_start >= -tolerance
        end_inside = value_end >= -tolerance
        if start_inside:
            clipped.append(start)
        if start_inside != end_inside:
            denominator = value_start - value_end
            if abs(denominator) > np.finfo(float).eps:
                fraction = value_start / denominator
                clipped.append(start + fraction * (end - start))

    # Adjacent duplicate intersections are removed to avoid degenerate polygons.
    cleaned = []
    for point in clipped:
        if not cleaned or not np.allclose(point, cleaned[-1], atol=1.0e-9, rtol=0):
            cleaned.append(point)
    if len(cleaned) > 1 and np.allclose(cleaned[0], cleaned[-1], atol=1.0e-9, rtol=0):
        cleaned.pop()
    return np.asarray(cleaned, dtype=float) if len(cleaned) >= 3 else np.empty((0, 2))


def interface_segment(
    triangle_xy: np.ndarray,
    interface_values: np.ndarray,
    level: float,
    tolerance: float = LEVEL_TOLERANCE_M,
) -> np.ndarray | None:
    """Intersect a triangular interface with a horizontal plane.

    Parameters
    ----------
    triangle_xy : ndarray, shape (3, 2)
        Triangle vertices in local XY coordinates, in metres.
    interface_values : ndarray, shape (3,)
        Interface elevations at the triangle vertices, in metres.
    level : float
        Horizontal section elevation, in metres.
    tolerance : float, optional
        Elevation tolerance for classifying vertices near the section plane.

    Returns
    -------
    ndarray, shape (2, 2), or None
        Segment endpoints in local XY coordinates, or None when fewer than
        two distinct intersections exist.
    """
    points = []
    for index in range(3):
        next_index = (index + 1) % 3
        start, end = triangle_xy[index], triangle_xy[next_index]
        value_start = float(interface_values[index] - level)
        value_end = float(interface_values[next_index] - level)
        if abs(value_start) <= tolerance:
            points.append(start)
        if (value_start < -tolerance and value_end > tolerance) or (
            value_start > tolerance and value_end < -tolerance
        ):
            fraction = value_start / (value_start - value_end)
            points.append(start + fraction * (end - start))

    unique = []
    for point in points:
        if not any(np.allclose(point, other, atol=1.0e-9, rtol=0) for other in unique):
            unique.append(point)
    if len(unique) < 2:
        return None
    if len(unique) == 2:
        return np.asarray(unique, dtype=float)
    # When the section contains a mesh edge, the farthest intersections define it.
    pair = max(
        ((i, j) for i in range(len(unique)) for j in range(i + 1, len(unique))),
        key=lambda ij: np.linalg.norm(unique[ij[0]] - unique[ij[1]]),
    )
    return np.asarray([unique[pair[0]], unique[pair[1]]], dtype=float)

## 4. Plot horizontal sections


In [ ]:
with plt.rc_context(PLOT_RC):
    fig, axes = plt.subplots(
        1,
        2,
        figsize=(FIGURE_WIDTH_CM / CM_PER_INCH, FIGURE_HEIGHT_CM / CM_PER_INCH),
        dpi=600,
        sharex=True,
        sharey=True,
        layout="constrained",
    )
    for panel_index, (ax, level, panel_label) in enumerate(
        zip(axes, ELEVATION_LEVELS_M, ("(a)", "(b)"))
    ):
        n_interfaces = interfaces_z.shape[1]
        n_bodies = n_interfaces - 1
        polygons_by_body = [[] for _ in range(n_bodies)]
        segments_by_interface = [[] for _ in range(n_interfaces)]
        for cell_index, triangle in enumerate(final_triangles):
            triangle_xy = final_nodes_xy[triangle]
            triangle_z = interfaces_z[triangle]
            if not np.isfinite(triangle_z).all():
                continue

            for body_index in range(n_bodies):
                if not cell_presence_mask[cell_index, body_index]:
                    continue
                # Two half-plane clips enforce lower <= level <= upper within each unit.
                # The resulting polygon is the horizontal intersection in local XY.
                polygon = clip_polygon_by_linear_field(
                    triangle_xy, level - triangle_z[:, body_index]
                )
                if len(polygon) >= 3:
                    coefficients = np.linalg.solve(
                        np.column_stack((triangle_xy, np.ones(3))),
                        np.asarray(triangle_z[:, body_index + 1], dtype=float),
                    )
                    upper_values = (
                        np.column_stack((polygon, np.ones(len(polygon)))) @ coefficients
                    )
                    polygon = clip_polygon_by_linear_field(
                        polygon, upper_values - level
                    )
                if len(polygon) >= 3:
                    x, y = np.asarray(polygon).T
                    area = 0.5 * abs(
                        float(np.dot(x, np.roll(y, -1)) - np.dot(y, np.roll(x, -1)))
                    )
                    if area > 1.0e-8:
                        polygons_by_body[body_index].append(polygon)

            for interface_index in range(n_interfaces):
                if interface_index == 0:
                    is_present = cell_presence_mask[cell_index, 0]
                elif interface_index == n_interfaces - 1:
                    is_present = cell_presence_mask[cell_index, -1]
                else:
                    is_present = (
                        cell_presence_mask[cell_index, interface_index - 1]
                        or cell_presence_mask[cell_index, interface_index]
                    )
                if not is_present:
                    continue
                segment = interface_segment(
                    triangle_xy, triangle_z[:, interface_index], level
                )
                if (
                    segment is not None
                    and np.linalg.norm(segment[1] - segment[0]) > 1.0e-8
                ):
                    segments_by_interface[interface_index].append(segment)

        # Section polygons use the configured unit colours and transparency.
        for body_index, polygons in enumerate(polygons_by_body):
            if polygons:
                ax.add_collection(
                    PolyCollection(
                        polygons,
                        facecolors=BODY_COLORS[body_index],
                        edgecolors="none",
                        linewidths=0,
                        alpha=0.9,
                        zorder=1,
                    )
                )

        # The shared mesh provides a spatial reference for the sections.
        ax.triplot(
            final_nodes_xy[:, 0],
            final_nodes_xy[:, 1],
            final_triangles,
            color="0.45",
            linewidth=0.3,
            alpha=0.5,
            zorder=2,
        )
        for segments in segments_by_interface:
            for segment in segments:
                ax.plot(
                    segment[:, 0],
                    segment[:, 1],
                    color="#3B3F45",
                    linewidth=0.5,
                    alpha=0.8,
                    zorder=3,
                )

        borehole_points = ax.scatter(
            initial_nodes_xy[:, 0],
            initial_nodes_xy[:, 1],
            s=8,
            facecolor="#1f5d8f",
            edgecolor="white",
            linewidth=0.5,
            zorder=4,
            label="Borehole location",
        )
        for name, (x, y) in zip(drill_names, initial_nodes_xy):
            ax.annotate(
                str(name),
                (x, y),
                xytext=(2.5, 2.5),
                textcoords="offset points",
                family=FONT_FAMILY,
                fontsize=4.2,
                color="0.15",
                zorder=5,
            )

        x_padding = np.ptp(final_nodes_xy[:, 0]) * 0.03
        y_padding = np.ptp(final_nodes_xy[:, 1]) * 0.03
        x_limits = (
            final_nodes_xy[:, 0].min() - x_padding,
            final_nodes_xy[:, 0].max() + x_padding,
        )
        y_limits = (
            final_nodes_xy[:, 1].min() - y_padding,
            final_nodes_xy[:, 1].max() + y_padding,
        )
        box_aspect = (y_limits[1] - y_limits[0]) / (x_limits[1] - x_limits[0])
        ax.set_xlim(*x_limits)
        ax.set_ylim(*y_limits)
        ax.set_box_aspect(box_aspect)
        ax.set_xlabel("X (m)", family=FONT_FAMILY, fontsize=7)
        ax.set_ylabel("Y (m)", family=FONT_FAMILY, fontsize=7)
        ax.set_axisbelow(True)

        ax.grid(axis="both", alpha=0.8, color="lightgray", linewidth=0.25)

        for spine in ax.spines.values():
            spine.set_linewidth(0.6)
            spine.set_color("#333333")

        ax.tick_params(
            axis="both",
            which="both",
            top=True,
            right=True,
            direction="in",
            labelsize=4,
            length=2,
            width=0.5,
        )
        ax.ticklabel_format(style="plain", useOffset=False)

        mesh_proxy = Line2D(
            [0], [0], color="0.45", linewidth=0.8, label="Shared triangle net"
        )
        handles = [
            Patch(facecolor=color, label=label)
            for label, color in reversed(list(zip(BODY_LABELS, BODY_COLORS)))
        ] + [
            mesh_proxy,
            borehole_points,
        ]
        legend = ax.legend(
            handles=handles,
            loc="upper right",
            borderaxespad=0.2,
            frameon=True,
            framealpha=0.9,
            edgecolor="0.5",
            fontsize=4,
            handlelength=1.5,
        )
        legend.get_frame().set_linewidth(0.4)
        plt.setp(legend.get_texts(), family=FONT_FAMILY)
        ax.text(
            0.015,
            0.985,
            f"{panel_label}",
            transform=ax.transAxes,
            va="top",
            family=FONT_FAMILY,
            fontsize=7,
            bbox=dict(facecolor="white", edgecolor="none", alpha=0.90, pad=0.8),
        )
        ax.tick_params(labelbottom=True, labelleft=panel_index == 0)
        if panel_index == 1:
            ax.set_ylabel("")
    plt.savefig(FIGURE_DIR / "horizontal section.png", dpi=600)
    plt.show()